<a href="https://colab.research.google.com/github/Davidsudheerkumar/24P31A4255_DriveReady_OPC_2026/blob/main/Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -q langchain langchain-groq gspread pydantic


In [2]:

import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("EduPredictAI Groq model is ready!")


EduPredictAI Groq model is ready!


In [3]:

from langchain.tools import tool

@tool
def calculate_percentage(marks_obtained: float, total_marks: float) -> str:
    """Calculate a student's percentage from obtained marks and total marks.

    Use this tool when someone asks for a student's percentage.

    Args:
        marks_obtained: Marks obtained by the student.
        total_marks: Maximum possible marks.
    """
    if total_marks <= 0:
        return "Error: Total marks must be greater than zero."

    if marks_obtained < 0 or marks_obtained > total_marks:
        return "Error: Obtained marks must be between zero and total marks."

    percentage = (marks_obtained / total_marks) * 100

    return f"Student percentage: {percentage:.2f}%"


@tool
def classify_performance(score: int) -> str:
    """Classify student performance using a score from 0 to 20.

    Use this tool when someone asks whether a student Passed, Failed or Dropped out.

    Args:
        score: Student performance score between 0 and 20.
    """
    if score < 0 or score > 20:
        return "Error: Score must be between 0 and 20."

    if score == 0:
        return "Dropout"

    elif score <= 9:
        return "Fail"

    else:
        return "Pass"


print("Tool 1:", calculate_percentage.name)
print("Tool 2:", classify_performance.name)


Tool 1: calculate_percentage
Tool 2: classify_performance


In [4]:

print(calculate_percentage.invoke({
    "marks_obtained": 450,
    "total_marks": 500
}))

print(classify_performance.invoke({
    "score": 15
}))


Student percentage: 90.00%
Pass


In [5]:

from typing import Optional
from pydantic import BaseModel, Field

class StudentRecord(BaseModel):
    """Academic record of an EduPredictAI student."""

    name: str = Field(
        description="Student's name"
    )

    subject: str = Field(
        description="Subject name"
    )

    marks: float = Field(
        description="Marks obtained by the student",
        ge=0
    )

    attendance: float = Field(
        description="Student attendance percentage",
        ge=0,
        le=100
    )

    date: str = Field(
        description="Date of the academic record"
    )

    concern: Optional[str] = Field(
        default=None,
        description="Academic concern or support required, if any"
    )


print("StudentRecord model created successfully!")


StudentRecord model created successfully!


In [6]:

extractor = llm.with_structured_output(StudentRecord)

messages = [
    "Hi, I am Ravi. I got 85 marks in maths out of 100, attendance is 92 percent today. No concerns.",

    "Priya here, maths marks 45, attendance 60%, recorded today. I need help with maths.",

    "My name is Arjun. I scored 78 in computer science and my attendance is 88 percent. Date is 27 Nov 2026.",

    "Hello, this is Sravani. I got 32 marks in English. Attendance is only 55 percent. I need academic support.",

    "Kiran, physics, 95 marks, attendance 98 percent, 27 November 2026, doing well."
]

student_records = []

for i, message in enumerate(messages, start=1):
    try:
        record = extractor.invoke(message)
        student_records.append(record)

        print(f"\nStudent {i}")
        print(record.model_dump())

    except Exception as e:
        print(f"Error processing student {i}: {e}")

print("\nTotal successful records:", len(student_records))



Student 1
{'name': 'Ravi', 'subject': 'maths', 'marks': 85.0, 'attendance': 92.0, 'date': '2026-11-27', 'concern': None}

Student 2
{'name': 'Priya', 'subject': 'maths', 'marks': 45.0, 'attendance': 60.0, 'date': '2026-11-27', 'concern': None}

Student 3
{'name': 'Arjun', 'subject': 'computer science', 'marks': 78.0, 'attendance': 88.0, 'date': '27 Nov 2026', 'concern': None}

Student 4
{'name': 'Sravani', 'subject': 'English', 'marks': 32.0, 'attendance': 55.0, 'date': '2026-11-27', 'concern': None}

Student 5
{'name': 'Kiran', 'subject': 'physics', 'marks': 95.0, 'attendance': 98.0, 'date': '27 November 2026', 'concern': 'doing well'}

Total successful records: 5


In [7]:

from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

sheet_name = "EduPredictAI Student Records"

try:
    sh = gc.open(sheet_name)
    print("Existing sheet opened!")

except gspread.SpreadsheetNotFound:
    sh = gc.create(sheet_name)
    print("New sheet created!")

ws = sh.sheet1

headers = [
    "name",
    "subject",
    "marks",
    "attendance",
    "date",
    "concern",
    "performance"
]

if not ws.get_all_values():
    ws.append_row(headers)

print("Google Sheet connected!")
print("Sheet URL:", sh.url)


Existing sheet opened!
Google Sheet connected!
Sheet URL: https://docs.google.com/spreadsheets/d/1hax3ih1vyq6G9FNZ7myLRRb3Ya2PsqjlRSA1Hoyl48Y


In [8]:

@tool
def log_student_record(
    name: str,
    subject: str,
    marks: float,
    attendance: float,
    date: str,
    concern: Optional[str] = None
) -> str:
    """Validate and save a student's academic record to EduPredictAI Google Sheets.

    Use only when the student's name, subject, marks, attendance and date are known.

    Args:
        name: Student's name.
        subject: Subject name.
        marks: Marks obtained, from 0 to 100.
        attendance: Attendance percentage, from 0 to 100.
        date: Date of the academic record.
        concern: Optional academic concern or support requirement.
    """

    try:
        student = StudentRecord(
            name=name,
            subject=subject,
            marks=marks,
            attendance=attendance,
            date=date,
            concern=concern
        )

        performance = classify_performance.invoke({
            "score": round(student.marks / 5)
        })

        if performance.startswith("Error"):
            return performance

        ws.append_row([
            student.name,
            student.subject,
            student.marks,
            student.attendance,
            student.date,
            student.concern or "None",
            performance
        ])

        return (
            f"Student record saved successfully! "
            f"Name: {student.name}, "
            f"Subject: {student.subject}, "
            f"Performance: {performance}."
        )

    except Exception as e:
        return f"Could not save student record: {e}"


print("Student record logging tool created!")


Student record logging tool created!


In [9]:

from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are the friendly EduPredictAI academic assistant.

PURPOSE:
Help teachers understand student academic performance.

RULES:
1. Always use calculate_percentage for percentage calculations.
2. Always use classify_performance for performance classification.
3. Use log_student_record only when all required student details are available.
4. Never invent student marks, attendance or dates.
5. Ask the user for missing information before saving.
6. Never claim a record was saved unless the tool confirms success.
7. Explain results in simple English.
"""

agent = create_agent(
    model=llm,
    tools=[
        calculate_percentage,
        classify_performance,
        log_student_record
    ],
    system_prompt=SYSTEM
)

print("EduPredictAI agent created successfully!")


EduPredictAI agent created successfully!


In [10]:

history = []

def chat(question):
    global history

    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": question}
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, something went wrong. Please try again."


print(chat("A student got 450 marks out of 500. Calculate the percentage."))

print(chat("A student scored 15 out of 20. Classify the performance."))

print(chat(
    "Save a student record. Name: David, Subject: Maths, "
    "Marks: 85, Attendance: 92, Date: 27 November 2026, "
    "Concern: None."
))


The student’s percentage is **90.00 %**.
That score of 15 out of 20 means the student **passed**.
✅ Student record saved successfully!  
- **Name:** David  
- **Subject:** Maths  
- **Marks:** 85/100  
- **Attendance:** 92%  
- **Date:** 27 November 2026  
- **Performance:** Pass  

Let me know if you’d like to add any additional details or analyze more records.


In [11]:

saved_count = 0

for record in student_records:
    result = log_student_record.invoke(record.model_dump())

    print(result)

    if result.startswith("Student record saved successfully"):
        saved_count += 1

print("\nTotal records saved:", saved_count)


Student record saved successfully! Name: Ravi, Subject: maths, Performance: Pass.
Student record saved successfully! Name: Priya, Subject: maths, Performance: Fail.
Student record saved successfully! Name: Arjun, Subject: computer science, Performance: Pass.
Student record saved successfully! Name: Sravani, Subject: English, Performance: Fail.
Student record saved successfully! Name: Kiran, Subject: physics, Performance: Pass.

Total records saved: 5


In [12]:

@tool("tool1")
def tool1(marks_obtained: float, total_marks: float) -> str:
    """Does stuff."""
    if total_marks <= 0:
        return "Invalid total marks."

    if marks_obtained < 0 or marks_obtained > total_marks:
        return "Invalid obtained marks."

    percentage = (marks_obtained / total_marks) * 100

    return f"Student percentage: {percentage:.2f}%"


print("Bad tool name:", tool1.name)
print("Good tool name:", calculate_percentage.name)


Bad tool name: tool1
Good tool name: calculate_percentage


In [13]:

bad_agent = create_agent(
    model=llm,
    tools=[tool1, classify_performance],
    system_prompt="Answer student academic questions using the available tools."
)

bad_result = bad_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Calculate the percentage for 450 marks out of 500."
    }]
})

for m in bad_result["messages"]:
    m.pretty_print()


================================ Human Message =================================

Calculate the percentage for 450 marks out of 500.
================================== Ai Message ==================================

The percentage is calculated as:

\[
\text{Percentage} = \frac{450}{500} \times 100 = 0.9 \times 100 = 90\%
\]

So, 450 marks out of 500 is **90 %**.


In [14]:

good_agent = create_agent(
    model=llm,
    tools=[calculate_percentage, classify_performance],
    system_prompt=SYSTEM
)

good_result = good_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Calculate the percentage for 450 marks out of 500."
    }]
})

for m in good_result["messages"]:
    m.pretty_print()


================================ Human Message =================================

Calculate the percentage for 450 marks out of 500.
================================== Ai Message ==================================
Tool Calls:
  calculate_percentage (fc_b93d9420-0c12-4c90-9683-441fdf11fadd)
 Call ID: fc_b93d9420-0c12-4c90-9683-441fdf11fadd
  Args:
    marks_obtained: 450
    total_marks: 500
================================= Tool Message =================================
Name: calculate_percentage

Student percentage: 90.00%
================================== Ai Message ==================================

You scored **450 out of 500**, which works out to a **90.00 %**. This means you earned 90 % of the total possible marks.


In [15]:

question = "Calculate the percentage for 450 marks out of 500."

trace_result = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

tool_calls = 0
model_calls = 0

print("========== AGENT TRACE ==========")

for message in trace_result["messages"]:

    message.pretty_print()

    if message.type == "ai":
        model_calls += 1
        tool_calls += len(message.tool_calls)

print("\n========== TRACE SUMMARY ==========")
print("Total model calls:", model_calls)
print("Total tool calls:", tool_calls)
print("Final answer:", trace_result["messages"][-1].content)


========== AGENT TRACE ==========
================================ Human Message =================================

Calculate the percentage for 450 marks out of 500.
================================== Ai Message ==================================
Tool Calls:
  calculate_percentage (fc_e824b94f-737f-418b-a9e5-3396908220b4)
 Call ID: fc_e824b94f-737f-418b-a9e5-3396908220b4
  Args:
    marks_obtained: 450
    total_marks: 500
================================= Tool Message =================================
Name: calculate_percentage

Student percentage: 90.00%
================================== Ai Message ==================================

You scored **450 out of 500**, which works out to a **90.00 %**. That means you earned 90 % of the available marks.

========== TRACE SUMMARY ==========
Total model calls: 2
Total tool calls: 1
Final answer: You scored **450 out of 500**, which works out to a **90.00 %**. That means you earned 90 % of the available marks.


Day - 4 Tasks


In [16]:

!pip install -q langchain langchain-groq gspread pydantic

import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("EduPredictAI AI model is ready!")


EduPredictAI AI model is ready!


In [17]:

from langchain.tools import tool

@tool
def calculate_percentage(marks_obtained: float, total_marks: float) -> str:
    """Calculate a student's percentage from obtained and total marks.

    Use this tool whenever a user asks to calculate student marks percentage.

    Args:
        marks_obtained: Marks obtained by the student.
        total_marks: Maximum possible marks.
    """
    if total_marks <= 0:
        return "Error: Total marks must be greater than zero."

    if marks_obtained < 0 or marks_obtained > total_marks:
        return "Error: Marks must be between zero and total marks."

    percentage = (marks_obtained / total_marks) * 100

    return f"Student percentage: {percentage:.2f}%"


@tool
def classify_performance(score: int) -> str:
    """Classify student performance as Pass, Fail or Dropout.

    Use this tool when a user asks about a student's performance.

    Args:
        score: Student score between 0 and 20.
    """
    if score < 0 or score > 20:
        return "Error: Score must be between 0 and 20."

    if score == 0:
        return "Dropout"
    elif score <= 9:
        return "Fail"
    else:
        return "Pass"


print(calculate_percentage.invoke({
    "marks_obtained": 450,
    "total_marks": 500
}))

print(classify_performance.invoke({
    "score": 15
}))


Student percentage: 90.00%
Pass


In [18]:

from typing import Optional
from pydantic import BaseModel, Field

class AdmissionEnquiry(BaseModel):
    """Student admission enquiry record for EduPredictAI."""

    student_name: str = Field(
        description="Full name of the student"
    )

    program: str = Field(
        description="Course or branch the student wants to join"
    )

    previous_percentage: float = Field(
        description="Student's previous academic percentage",
        ge=0,
        le=100
    )

    entrance_score: Optional[int] = Field(
        default=None,
        description="Entrance examination score, if available"
    )

    enquiry_date: str = Field(
        description="Date of the admission enquiry"
    )

    question: Optional[str] = Field(
        default=None,
        description="Student's admission question or requirement"
    )


print("AdmissionEnquiry Pydantic model created!")


AdmissionEnquiry Pydantic model created!


In [19]:

extractor = llm.with_structured_output(AdmissionEnquiry)

messages = [
    "Hi I'm Ravi, want CSE AIML admission. I got 92% in intermediate and my entrance score is 850. Today is 27 Nov 2026. Tell me about fees.",

    "hello mam this is Priya. interested in ECE branch, inter percentage 85.5, no entrance exam yet. enquiry on 27 November 2026.",

    "My name Arjun, I need admission in computer science, got 78 percent in previous studies, entrance marks 720. Date 27 Nov 2026. What are the eligibility rules?",

    "Hi, Sravani here. I want to join IT. My previous percentage is 88. Entrance score not available. I want to know about scholarships. Today 27 November 2026.",

    "This is Kiran, looking for AI and ML course, scored 95% in intermediate, entrance score 910, asking about admission process, 27 November 2026."
]

admission_records = []

for i, message in enumerate(messages, start=1):
    try:
        record = extractor.invoke(message)
        admission_records.append(record)

        print(f"\nAdmission Enquiry {i}")
        print(record.model_dump())

    except Exception as e:
        print(f"Error in message {i}: {e}")

print("\nSuccessfully extracted records:", len(admission_records))



Admission Enquiry 1
{'student_name': 'Ravi', 'program': 'CSE AIML', 'previous_percentage': 92.0, 'entrance_score': 850, 'enquiry_date': '2026-11-27', 'question': 'Tell me about fees.'}

Admission Enquiry 2
{'student_name': 'Priya', 'program': 'ECE', 'previous_percentage': 85.5, 'entrance_score': None, 'enquiry_date': '27 November 2026', 'question': None}

Admission Enquiry 3
{'student_name': 'Arjun', 'program': 'Computer Science', 'previous_percentage': 78.0, 'entrance_score': 720, 'enquiry_date': '27 Nov 2026', 'question': 'What are the eligibility rules?'}

Admission Enquiry 4
{'student_name': 'Sravani', 'program': 'IT', 'previous_percentage': 88.0, 'entrance_score': None, 'enquiry_date': '2026-11-27', 'question': 'I want to know about scholarships.'}

Admission Enquiry 5
{'student_name': 'Kiran', 'program': 'AI and ML', 'previous_percentage': 95.0, 'entrance_score': 910, 'enquiry_date': '2026-11-27', 'question': 'admission process'}

Successfully extracted records: 5


In [20]:

from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

sheet_name = "EduPredictAI Admission Enquiries"

try:
    sh = gc.open(sheet_name)
    print("Existing Google Sheet opened!")

except gspread.SpreadsheetNotFound:
    sh = gc.create(sheet_name)
    print("New Google Sheet created!")

ws = sh.sheet1

headers = [
    "Student Name",
    "Program",
    "Previous Percentage",
    "Entrance Score",
    "Enquiry Date",
    "Question"
]

if not ws.get_all_values():
    ws.append_row(headers)

print("Google Sheet connected successfully!")
print("Sheet URL:", sh.url)


New Google Sheet created!
Google Sheet connected successfully!
Sheet URL: https://docs.google.com/spreadsheets/d/1tYh8OPkVhyKUt5iE-KGdFq07g8qPKCce39y33WLWXmM


In [21]:

@tool
def log_admission_enquiry(
    student_name: str,
    program: str,
    previous_percentage: float,
    enquiry_date: str,
    entrance_score: Optional[int] = None,
    question: Optional[str] = None
) -> str:
    """Validate and save a student admission enquiry to Google Sheets.

    Use only when the student's name, program, previous percentage
    and enquiry date are available.

    Args:
        student_name: Full name of the student.
        program: Course or branch requested.
        previous_percentage: Previous academic percentage from 0 to 100.
        enquiry_date: Date of the enquiry.
        entrance_score: Optional entrance examination score.
        question: Optional admission question from the student.
    """

    try:
        enquiry = AdmissionEnquiry(
            student_name=student_name,
            program=program,
            previous_percentage=previous_percentage,
            enquiry_date=enquiry_date,
            entrance_score=entrance_score,
            question=question
        )

        ws.append_row([
            enquiry.student_name,
            enquiry.program,
            enquiry.previous_percentage,
            enquiry.entrance_score if enquiry.entrance_score is not None else "",
            enquiry.enquiry_date,
            enquiry.question or "None"
        ])

        return (
            f"Admission enquiry saved successfully! "
            f"Student: {enquiry.student_name}, "
            f"Program: {enquiry.program}."
        )

    except Exception as e:
        return f"Could not save admission enquiry: {e}"


print("log_admission_enquiry tool created!")


log_admission_enquiry tool created!


In [22]:

from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are the friendly EduPredictAI admission assistant.

RULES:
1. Use calculate_percentage for percentage calculations.
2. Use classify_performance for Pass, Fail and Dropout classification.
3. Use log_admission_enquiry to save complete admission enquiries.
4. Never invent student information.
5. Ask for missing required information before saving.
6. Confirm the result after saving.
7. Reply in simple English.
"""

agent = create_agent(
    model=llm,
    tools=[
        calculate_percentage,
        classify_performance,
        log_admission_enquiry
    ],
    system_prompt=SYSTEM
)

print("EduPredictAI agent created successfully!")


EduPredictAI agent created successfully!


In [23]:

history = []

def chat(question):
    global history

    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": question}
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, something went wrong. Please try again."


print(chat("I got 450 marks out of 500. Calculate my percentage."))

print(chat("A student scored 7 out of 20. Classify the performance."))

print(chat(
    "My name is David. I want CSE AIML admission. "
    "My previous percentage is 90. Entrance score is 850. "
    "My enquiry date is 27 November 2026. "
    "I want to know about scholarships. Save my enquiry."
))


Your percentage is 90.00%.
The student has **Failed**.
Your enquiry has been saved.  
**Student:** David  
**Program:** CSE AIML  
**Previous Percentage:** 90%  
**Entrance Score:** 850  
**Enquiry Date:** 27 November 2026  
**Question:** I want to know about scholarships  

We’ll get back to you soon with scholarship details. If you need anything else, just let us know!


In [24]:

saved_count = 0

for record in admission_records:

    result = log_admission_enquiry.invoke(
        record.model_dump()
    )

    print(result)

    if result.startswith("Admission enquiry saved successfully"):
        saved_count += 1

print("\nTotal admission enquiries saved:", saved_count)


Admission enquiry saved successfully! Student: Ravi, Program: CSE AIML.
Admission enquiry saved successfully! Student: Priya, Program: ECE.
Admission enquiry saved successfully! Student: Arjun, Program: Computer Science.
Admission enquiry saved successfully! Student: Sravani, Program: IT.
Admission enquiry saved successfully! Student: Kiran, Program: AI and ML.

Total admission enquiries saved: 5


In [25]:

@tool("tool1")
def tool1(x: float, n: float) -> str:
    """does stuff"""
    if n <= 0:
        return "Invalid total."

    if x < 0 or x > n:
        return "Invalid marks."

    return f"Percentage: {(x/n)*100:.2f}%"


print("Tool name:", tool1.name)
print("Tool description:", tool1.description)


Tool name: tool1
Tool description: does stuff


In [26]:

bad_agent = create_agent(
    model=llm,
    tools=[
        tool1,
        classify_performance,
        log_admission_enquiry
    ],
    system_prompt="You are an academic assistant. Use available tools to answer student questions."
)

bad_result = bad_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Calculate my percentage. I got 450 marks out of 500."
    }]
})

for m in bad_result["messages"]:
    m.pretty_print()


================================ Human Message =================================

Calculate my percentage. I got 450 marks out of 500.
================================== Ai Message ==================================

Your percentage is:

\[
\frac{450}{500} \times 100 = 90\%
\]

So you scored **90 %**.


In [27]:

good_agent = create_agent(
    model=llm,
    tools=[
        calculate_percentage,
        classify_performance,
        log_admission_enquiry
    ],
    system_prompt=SYSTEM
)

good_result = good_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Calculate my percentage. I got 450 marks out of 500."
    }]
})

for m in good_result["messages"]:
    m.pretty_print()


================================ Human Message =================================

Calculate my percentage. I got 450 marks out of 500.
================================== Ai Message ==================================
Tool Calls:
  calculate_percentage (fc_a85870db-8712-4c17-9ee8-42de3f9374a5)
 Call ID: fc_a85870db-8712-4c17-9ee8-42de3f9374a5
  Args:
    marks_obtained: 450
    total_marks: 500
================================= Tool Message =================================
Name: calculate_percentage

Student percentage: 90.00%
================================== Ai Message ==================================

Your percentage is 90.00%.


In [28]:

trace_result = good_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Calculate the percentage for 450 marks out of 500."
    }]
})

tool_calls = 0
model_calls = 0

print("========== AGENT TRACE ==========")

for message in trace_result["messages"]:

    message.pretty_print()

    if message.type == "ai":
        model_calls += 1
        tool_calls += len(message.tool_calls)

print("\n========== TRACE SUMMARY ==========")
print("Total model calls:", model_calls)
print("Total tool calls:", tool_calls)
print("Final answer:", trace_result["messages"][-1].content)


========== AGENT TRACE ==========
================================ Human Message =================================

Calculate the percentage for 450 marks out of 500.
================================== Ai Message ==================================
Tool Calls:
  calculate_percentage (fc_439a31f3-7d0d-486b-8a58-9b3518e4f724)
 Call ID: fc_439a31f3-7d0d-486b-8a58-9b3518e4f724
  Args:
    marks_obtained: 450
    total_marks: 500
================================= Tool Message =================================
Name: calculate_percentage

Student percentage: 90.00%
================================== Ai Message ==================================

The percentage is 90.00%.

========== TRACE SUMMARY ==========
Total model calls: 2
Total tool calls: 1
Final answer: The percentage is 90.00%.
